# 03 · Exploratory Data Analysis — Doomscroll

**Goal:** understand behaviour, engagement and temporal structure well enough to justify the intent-aware features and model choices.

| Section | Question it answers |
|---|---|
| 1 Overview | How big, how sparse, what is the intent mix? |
| 2–3 Actions | How rare is each action; how do actions relate? (justifies tiering) |
| 4–5 Users & items | Long tails, inequality, popularity bias |
| 6 Categories | Where does attention go; how concentrated are user interests? |
| 7–8 Time & sessions | Daily/weekly rhythm, session structure |
| 9 Watch behaviour | Does watch time carry intent beyond clicks? |
| 10 Repeat & recency | Do recency/repeat features carry signal? |
| 11 Search vs feed | Does explicit search intent predict feed engagement? |
| 12 Cohorts | Retention in the time window |
| 13 Cold-start & drift | How hard is the temporal split? |
| 14 Feature viability | Do engineered features beat chance? Which ones? |
| 15 Interest representation | Can users be separated by interest vectors? |

Figures are saved to `eda/figures/03_*.png`. Run notebooks 01 and 02 first.

In [ ]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})

ROOT = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
PROC, FIG = ROOT / "data" / "processed", ROOT / "eda" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
SEED = 42
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"03_{name}.png", dpi=150, bbox_inches="tight"); plt.show()

def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return float((2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n)
def lorenz(x):
    x = np.sort(np.asarray(x, float)); return np.insert(np.cumsum(x) / x.sum(), 0, 0)

cfg = json.load(open(PROC / "preprocess_config.json"))
ev = pd.read_parquet(PROC / "events_enriched.parquet")
ev["cat"] = ev["cat"].astype(str); ev["source"] = ev["source"].astype(str)
flags = [f for f in cfg["flags"] if ev[f].sum() > 0]
S = ev.sample(min(len(ev), 1_000_000), random_state=SEED)       # sample for heavy plots
rng = np.random.default_rng(SEED)
U_SAMPLE = rng.choice(ev["user"].unique(), size=min(20_000, ev["user"].nunique()), replace=False)
print(f"{len(ev):,} events | flags: {flags}")

## 1 · Overview

In [ ]:
nu, ni = ev["user"].nunique(), ev["item"].nunique()
kpi = pd.Series({"events": len(ev), "users": nu, "items": ni, "categories": ev["cat"].nunique(),
                 "days covered": round(float(ev["t"].max() - ev["t"].min()), 1),
                 "feed share %": round((ev["source"] == "rec").mean() * 100, 1), "search share %": round((ev["source"] == "search").mean() * 100, 1),
                 "events with any engagement %": round(ev["pos"].mean() * 100, 2),
                 "user–item matrix density %": round(ev[["user", "item"]].drop_duplicates().shape[0] / (nu * ni) * 100, 5),
                 "median events / user": ev.groupby("user").size().median(), "median events / item": ev.groupby("item").size().median()})
display(kpi.to_frame("value"))

tier_src = ev.groupby(["source", "tier"]).size().unstack(fill_value=0)
(tier_src.div(tier_src.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, figsize=(7, 4), colormap="viridis")
plt.title("Intent-tier mix by source (%)"); plt.ylabel("%"); plt.xticks(rotation=0); plt.legend(title="tier", bbox_to_anchor=(1, 1))
savefig("tier_by_source")

## 2 · Action rates and the engagement funnel

In [ ]:
rates = ev.groupby("source")[flags].mean().T * 100
cond = {f: ev.loc[ev["click"] == 1, f].mean() * 100 for f in flags if f != "click"} if "click" in flags else {}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
rates.plot(kind="barh", ax=ax[0]); ax[0].set(title="P(action) by source (%)", xlabel="%"); ax[0].set_xscale("log")
if cond:
    pd.Series(cond).sort_values().plot(kind="barh", ax=ax[1], color="teal"); ax[1].set(title="P(action | click) — depth of engagement (%)", xlabel="%")
savefig("action_rates")
display(rates.round(3))

## 3 · How do actions relate? (justifies graded intent instead of one 'positive' class)

In [ ]:
M = S[flags].astype("float32").to_numpy()
co = M.T @ M
cond_m = pd.DataFrame(co / np.maximum(np.diag(co)[:, None], 1), index=flags, columns=flags)
phi = pd.DataFrame(np.corrcoef(M.T), index=flags, columns=flags)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cond_m, annot=True, fmt=".2f", cmap="Blues", ax=ax[0]); ax[0].set(title="P(column | row)  — read: given row action, chance of column action")
sns.heatmap(phi, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[1]); ax[1].set(title="Phi correlation between actions")
savefig("action_cooccurrence")

## 4 · User activity: long tail and inequality

In [ ]:
upe = ev.groupby("user").size()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.log10(upe), bins=50, color="steelblue"); ax[0].set(title="Events per user (log10)", xlabel="log10(events)", ylabel="users")
ax[1].plot(np.linspace(0, 1, len(upe) + 1), lorenz(upe), label=f"Gini = {gini(upe):.2f}"); ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set(title="Lorenz curve: user activity", xlabel="cumulative share of users", ylabel="cumulative share of events"); ax[1].legend()
adays = ev.groupby("user")["day"].nunique(); adays.plot(kind="hist", bins=range(1, int(adays.max()) + 2), ax=ax[2], color="coral", rwidth=.9)
ax[2].set(title="Active days per user", xlabel="days")
savefig("user_activity")
top = upe.sort_values(ascending=False)
print(f"top 1% of users produce {top.head(max(1, len(top)//100)).sum() / top.sum():.1%} of events | top 10%: {top.head(len(top)//10).sum() / top.sum():.1%}")

## 5 · Item popularity and popularity bias

In [ ]:
ipe = ev.groupby("item").size().sort_values(ascending=False)
ist = ev.groupby("item").agg(n=("user", "size"), pos=("pos", "mean"), score=("score", "mean"))
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].loglog(np.arange(1, len(ipe) + 1), ipe.values); ax[0].set(title="Item rank–frequency (log–log)", xlabel="rank", ylabel="events")
ax[1].plot(np.linspace(0, 1, len(ipe) + 1), lorenz(ipe), label=f"Gini = {gini(ipe):.2f}"); ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set(title="Lorenz curve: item exposure", xlabel="share of items", ylabel="share of events"); ax[1].legend()
d = ist[ist["n"] >= 20]
hb = ax[2].hexbin(d["n"], d["pos"], xscale="log", gridsize=40, cmap="magma", mincnt=1); plt.colorbar(hb, ax=ax[2])
ax[2].set(title="Exposure vs engagement rate (items ≥20 events)", xlabel="exposures (log)", ylabel="engagement rate")
savefig("item_popularity")
print(f"top 1% of items get {ipe.head(max(1, len(ipe)//100)).sum() / ipe.sum():.1%} of exposure | corr(log exposure, engagement) = {np.corrcoef(np.log10(d['n']), d['pos'])[0,1]:.3f}")

## 6 · Categories: attention, engagement and interest concentration

In [ ]:
cs = ev.groupby("cat").agg(events=("user", "size"), pos_rate=("pos", "mean"), mean_score=("score", "mean"), users=("user", "nunique")).sort_values("events", ascending=False)
top_cats = cs.head(15).index
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
cs.head(15)["events"].plot(kind="barh", ax=ax[0], color="slateblue"); ax[0].invert_yaxis(); ax[0].set(title="Top categories by exposure", xlabel="events")
sc = ax[1].scatter(cs["events"] / cs["events"].sum() * 100, cs["pos_rate"] * 100, s=cs["users"] / cs["users"].max() * 400 + 20, alpha=.6, c=cs["mean_score"], cmap="viridis")
for c_, r in cs.head(8).iterrows(): ax[1].annotate(c_, (r["events"] / cs["events"].sum() * 100, r["pos_rate"] * 100), fontsize=8)
ax[1].set(title="Exposure share vs engagement rate (size = users, colour = mean intent)", xlabel="exposure share %", ylabel="engagement rate %"); plt.colorbar(sc, ax=ax[1])
savefig("category_overview")

rate_cat = ev[ev["cat"].isin(top_cats)].groupby("cat")[flags].mean() * 100
plt.figure(figsize=(9, 6)); sns.heatmap((rate_cat - rate_cat.mean()) / rate_cat.std(), annot=rate_cat.round(1), fmt="", cmap="RdBu_r", center=0, cbar_kws={"label": "z-score per action"})
plt.title("Category × action rate (%, colour = z-score within action)"); savefig("category_action_heatmap")

ue = ev[ev["pos"] == 1].groupby(["user", "cat"]).size().rename("n").reset_index()
ue["p"] = ue["n"] / ue.groupby("user")["n"].transform("sum")
ent = ue.groupby("user")["p"].apply(lambda p: -(p * np.log2(p)).sum()); ncat = ue.groupby("user")["cat"].nunique()
top1 = ue.groupby("user")["p"].max()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ent.plot(kind="hist", bins=40, ax=ax[0], color="darkcyan"); ax[0].set(title="Interest entropy per user (bits)", xlabel="entropy")
ncat.plot(kind="hist", bins=range(1, int(ncat.max()) + 2), ax=ax[1], color="goldenrod", rwidth=.9); ax[1].set(title="Distinct categories engaged per user")
top1.plot(kind="hist", bins=40, ax=ax[2], color="indianred"); ax[2].set(title="Share of engagement in user's #1 category", xlabel="share")
savefig("interest_concentration")
print(f"median share in #1 category: {top1.median():.2f} | median categories engaged: {ncat.median():.0f}")

## 7 · Temporal patterns

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
daily = ev.groupby(["day", "source"]).size().unstack(fill_value=0); daily.plot(ax=ax[0, 0], marker="o"); ax[0, 0].set(title="Events per day (dashed = engagement rate)", xlabel="day index", ylabel="events"); ax[0, 0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
dr = ev.groupby("day")["pos"].mean() * 100; ax2 = ax[0, 0].twinx(); dr.plot(ax=ax2, color="k", ls="--"); ax2.set_ylabel("engagement rate %"); ax2.grid(False)
hm = ev.pivot_table(index="dow", columns="hour", values="user", aggfunc="size", fill_value=0)
hm.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][: len(hm)] if len(hm) == 7 else hm.index
sns.heatmap(hm, cmap="YlOrRd", ax=ax[0, 1]); ax[0, 1].set(title="Volume: weekday × local hour", xlabel="hour", ylabel="")
hr = ev.groupby("hour").agg(vol=("user", "size"), pos=("pos", "mean"), score=("score", "mean"))
ax[1, 0].bar(hr.index, hr["vol"], color="lightsteelblue"); ax[1, 0].set(title="Volume and engagement by hour", xlabel="local hour", ylabel="events")
b = ax[1, 0].twinx(); b.plot(hr.index, hr["pos"] * 100, color="crimson", marker="o"); b.set_ylabel("engagement rate %"); b.grid(False)
tm = ev.groupby(["hour", "tier"]).size().unstack(fill_value=0); (tm.div(tm.sum(axis=1), axis=0) * 100).plot(kind="area", stacked=True, ax=ax[1, 1], colormap="viridis", alpha=.85)
ax[1, 1].set(title="Intent-tier mix across the day (%)", xlabel="local hour", ylabel="%"); ax[1, 1].legend(title="tier", loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
savefig("temporal_patterns")
wk = ev.assign(weekend=ev["dow"] >= 5).groupby("weekend").agg(events=("user", "size"), pos=("pos", "mean"), score=("score", "mean")); display(wk)

## 8 · Sessions and inter-event gaps
Session = run of events with <30 min gap (computed on a 20k-user sample). *Caveat:* the logs contain interactions the dataset kept, so gaps are lower-bounded by what was logged.

In [ ]:
sub = ev[ev["user"].isin(U_SAMPLE)].sort_values(["user", "ts"])
gap = sub.groupby("user")["ts"].diff().dt.total_seconds() / 60
sub["sess"] = ((gap.isna()) | (gap > 30)).cumsum()
sess = sub.groupby("sess").agg(user=("user", "first"), n=("item", "size"), dur_min=("ts", lambda s: (s.max() - s.min()).total_seconds() / 60), pos=("pos", "mean"), day=("day", "first"))
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
ax[0].hist(np.log10(gap.dropna().clip(lower=1 / 60)), bins=60, color="slategray"); ax[0].set(title="Gap between consecutive events", xlabel="log10(minutes)")
ax[1].hist(sess["n"].clip(upper=sess["n"].quantile(.99)), bins=40, color="teal"); ax[1].set(title="Events per session")
ax[2].hist(np.log10(sess["dur_min"].clip(lower=0.01)), bins=40, color="orchid"); ax[2].set(title="Session length", xlabel="log10(min)")
sess.groupby(["user", "day"]).size().plot(kind="hist", bins=range(1, 15), ax=ax[3], rwidth=.9, color="orange"); ax[3].set(title="Sessions per active user-day")
savefig("sessions")
sess["n_bin"] = pd.cut(sess["n"], [0, 1, 3, 10, 30, 100, 1e9], labels=["1", "2-3", "4-10", "11-30", "31-100", "100+"])
display(sess.groupby("n_bin", observed=True).agg(sessions=("n", "size"), engagement_rate=("pos", "mean")).round(3))
print(f"{len(sess):,} sessions | median events/session {sess['n'].median():.0f} | median session {sess['dur_min'].median():.1f} min")

## 9 · Watch behaviour

In [ ]:
if "play_ratio" in ev and ev["play_ratio"].notna().any():
    w = S.dropna(subset=["play_ratio"])
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
    ax[0].hist(w["play_ratio"].clip(upper=3), bins=60, color="mediumseagreen"); ax[0].set(title="Watch ratio (play_time / duration, clipped at 3)", xlabel="ratio")
    sns.boxplot(data=w, x="tier", y="play_ratio", ax=ax[1], showfliers=False, palette="viridis"); ax[1].set(title="Watch ratio by intent tier")
    w["dur_bin"] = pd.qcut(w["duration"].rank(method="first"), 10, labels=False) if "duration" in w and w["duration"].notna().any() else 0
    g = w.groupby("dur_bin").agg(pos=("pos", "mean"), ratio=("play_ratio", "mean"))
    g["pos"].plot(ax=ax[2], marker="o", label="engagement rate"); ax[2].set(title="Duration decile → engagement / watch ratio", xlabel="duration decile (short → long)"); ax[2].legend(loc="upper left")
    ax[2].twinx().plot(g.index, g["ratio"], color="crimson", marker="s"); 
    savefig("watch_behaviour")
    print("corr(play_ratio, score) =", round(w[["play_ratio", "score"]].corr().iloc[0, 1], 3))
else:
    print("No play_ratio available -> skipping watch-behaviour analysis.")

## 10 · Repeat interactions, novelty and recency decay

In [ ]:
pairs = sub.groupby(["user", "item"]).size()
sub["first_cat"] = sub.groupby(["user", "cat"])["t"].transform("min"); sub["is_new_cat"] = (sub["t"] == sub["first_cat"]).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
pairs.clip(upper=10).value_counts().sort_index().plot(kind="bar", ax=ax[0], color="steelblue"); ax[0].set(title="Times a user saw the same video (10 = 10+)", xlabel="", ylabel="user–item pairs", yscale="log")
nov = sub.groupby("day").agg(new_cat=("is_new_cat", "mean")); (nov * 100).plot(ax=ax[1], marker="o", legend=False); ax[1].set(title="Share of events in a category new to the user", xlabel="day", ylabel="%")
Xtr = pd.read_parquet(PROC / "features_train.parquet"); Xs = Xtr.sample(min(len(Xtr), 800_000), random_state=SEED)
r = Xs[Xs["cold_uc"] == 0].assign(rb=lambda d: pd.cut(d["uc_recency"], [-.01, .25, .5, 1, 2, 3, 5, 7.1])).groupby("rb", observed=True)["label_pos"].agg(["mean", "size"])
ax[2].plot(range(len(r)), r["mean"] * 100, marker="o", color="darkorange"); ax[2].set_xticks(range(len(r))); ax[2].set_xticklabels([str(i) for i in r.index], rotation=30, fontsize=8)
ax[2].set(title="Engagement rate vs days since last event in same category", ylabel="engagement %")
savefig("repeat_novelty_recency")
rp = Xs.assign(p=Xs["ui_prior_n"].clip(upper=3)).groupby("p")["label_pos"].agg(["mean", "size"]).round(4); rp.index = ["0", "1", "2", "3+"][: len(rp)]
print("Engagement rate by # prior views of the same video:"); display(rp)
print(f"{(pairs > 1).mean():.1%} of user–video pairs repeat | median share new-category events: {nov['new_cat'].median():.1%}")

## 11 · Search vs feed
Does a user's search history predict feed engagement in the same category? We take the first half of the window as history and test the second half.

In [ ]:
mid = ev["t"].median()
hist, fut = ev[ev["t"] < mid], ev[(ev["t"] >= mid) & (ev["source"] == "rec")]
searched = hist[(hist["source"] == "search") & (hist["pos"] == 1)][["user", "cat"]].drop_duplicates().assign(searched=1)
engaged = hist[(hist["source"] == "rec") & (hist["pos"] == 1)].groupby(["user", "cat"]).size().rename("hist_feed_pos").reset_index()
f = fut.merge(searched, on=["user", "cat"], how="left").merge(engaged, on=["user", "cat"], how="left")
f["searched"] = f["searched"].fillna(0); f["fed"] = (f["hist_feed_pos"].fillna(0) > 0).astype(int)
lift = f.groupby(["fed", "searched"]).agg(events=("user", "size"), eng=("pos", "mean"), mean_tier=("tier", "mean")).round(4); display(lift)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ev.groupby(["source", "hour"]).size().unstack(0).pipe(lambda d: d / d.sum() * 100).plot(ax=ax[0]); ax[0].set(title="Hourly profile: feed vs search (% of source)", xlabel="hour")
pq = ev[ev["source"] == "search"].groupby("user").size(); ax[1].hist(np.log10(pq), bins=40, color="slateblue"); ax[1].set(title="Search events per user (log10)")
lift["eng"].unstack().plot(kind="bar", ax=ax[2]); ax[2].set(title="Feed engagement rate: past feed engagement (fed) × past search (searched)", xlabel="fed in category"); ax[2].legend(title="searched")
savefig("search_vs_feed")
print(f"{(ev['source'] == 'search').mean():.1%} of events are search | users with ≥1 search: {pq.shape[0] / ev['user'].nunique():.1%}")

## 12 · Cohort retention within the window

In [ ]:
first = ev.groupby("user")["day"].min().rename("first_day")
act = ev[["user", "day"]].drop_duplicates().join(first, on="user"); act["off"] = act["day"] - act["first_day"]
coh = act.groupby(["first_day", "off"])["user"].nunique().unstack(fill_value=0)
size = coh[0]; ret = coh.div(size, axis=0) * 100
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(ret, cmap="Blues", ax=ax[0], annot=ret.shape[0] <= 20, fmt=".0f", annot_kws={"size": 7}); ax[0].set(title="Retention % by first-active day × days since", xlabel="days since first seen", ylabel="first-active day")
(ev.groupby("day")["user"].nunique()).plot(kind="bar", ax=ax[1], color="cadetblue"); ax[1].set(title="Daily active users", xlabel="day")
savefig("cohorts")

## 13 · Cold-start and distribution drift across splits

In [ ]:
parts = {s: pd.read_parquet(PROC / f"features_{s}.parquet", columns=["user", "label_tier", "label_pos", "label_strong", "ctx_is_search", "cold_user", "cold_item", "cold_uc"]) for s in ("train", "val", "test")}
dr = pd.DataFrame({s: dict(rows=len(d), pos_rate=d["label_pos"].mean() * 100, strong_rate=d["label_strong"].mean() * 100, search_share=d["ctx_is_search"].mean() * 100,
                           cold_user=d["cold_user"].mean() * 100, cold_item=d["cold_item"].mean() * 100, cold_user_cat=d["cold_uc"].mean() * 100) for s, d in parts.items()}).T.round(2)
display(dr)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
dr[["cold_user", "cold_item", "cold_user_cat"]].plot(kind="bar", ax=ax[0]); ax[0].set(title="Cold-start share of label rows (%)"); ax[0].tick_params(axis="x", rotation=0)
td = pd.concat({s: d["label_tier"].value_counts(normalize=True).sort_index() * 100 for s, d in parts.items()}, axis=1); td.T.plot(kind="bar", stacked=True, ax=ax[1], colormap="viridis")
ax[1].set(title="Label-tier mix per split (%) – drift check"); ax[1].tick_params(axis="x", rotation=0); ax[1].legend(title="tier")
savefig("coldstart_drift")

## 14 · Feature viability
Single-feature ROC-AUC for predicting (a) any engagement and (b) strong engagement (like/comment/share/follow) on **train** rows. AUC = 0.5 is chance; values far from 0.5 (either side) carry signal. This is not a model, just a sanity check that the engineered features are informative and which groups matter.

In [ ]:
F = Xtr.sample(min(len(Xtr), 500_000), random_state=SEED)
cols = [c for c in F.columns if re.match(r"^(u|uc|i|ui|ctx|cold)_", c) and F[c].nunique() > 1]
rows = []
for c in cols:
    x = F[c].astype(float)
    rows.append((c, roc_auc_score(F["label_pos"], x), roc_auc_score(F["label_strong"], x) if F["label_strong"].nunique() > 1 else np.nan))
auc = pd.DataFrame(rows, columns=["feature", "auc_pos", "auc_strong"]); auc["edge"] = (auc["auc_pos"] - .5).abs()
auc = auc.sort_values("edge", ascending=False).drop(columns="edge")
auc["group"] = auc["feature"].str.extract(r"^(uc|ui|u|i|ctx|cold)")[0]
top = auc.head(20).iloc[::-1]
fig, ax = plt.subplots(1, 2, figsize=(15, 6.5))
y = np.arange(len(top)); ax[0].barh(y - .2, top["auc_pos"] - .5, .4, label="any engagement", left=.5); ax[0].barh(y + .2, top["auc_strong"] - .5, .4, label="strong engagement", left=.5)
ax[0].axvline(.5, color="k", lw=1); ax[0].set_yticks(y); ax[0].set_yticklabels(top["feature"]); ax[0].set(title="Top-20 single-feature AUC (bars start at 0.5 = chance)", xlabel="ROC-AUC"); ax[0].legend()
sns.boxplot(data=auc.assign(edge=(auc["auc_pos"] - .5).abs()), x="group", y="edge", ax=ax[1], palette="Set2"); ax[1].set(title="Signal strength by feature group (|AUC−0.5|)", ylabel="|AUC − 0.5|")
savefig("feature_viability")
display(auc.head(15).round(3))
best = auc.iloc[0]; print(f"Best single feature: {best['feature']} (AUC any={best['auc_pos']:.3f}, strong={best['auc_strong']:.3f})")

In [ ]:
key = [c for c in ["uc_decay_share", "uc_lift", "u_pos_rate", "i_pos_rate", "ui_prior_n"] if c in F]
fig, ax = plt.subplots(1, len(key), figsize=(4 * len(key), 3.8))
for a, c in zip(np.atleast_1d(ax), key):
    q = F.assign(b=pd.qcut(F[c].rank(method="first"), 10, labels=False)).groupby("b")[["label_pos", "label_strong"]].mean() * 100
    q.plot(ax=a, marker="o"); a.set(title=c, xlabel="decile of feature", ylabel="% rate"); a.legend(fontsize=7)
fig.suptitle("Engagement rate by feature decile (monotone rising = useful, intent-aware signal)", y=1.03); savefig("feature_deciles")

## 15 · User-interest representation

In [ ]:
ui = pd.read_parquet(PROC / "user_interest.parquet"); ui = ui[ui["split"] == "train"]
top12 = ev["cat"].value_counts().head(12).index
P = ui.pivot_table(index="user", columns="cat", values="uc_decay_share", fill_value=0)
samp = P.sample(min(len(P), 25), random_state=SEED)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(samp[[c for c in top12 if c in samp]], cmap="mako_r", ax=ax[0], cbar_kws={"label": "decayed intent share"}); ax[0].set(title="Interest vectors of 25 random users (top-12 categories)", ylabel="user")
Pm = P.sample(min(len(P), 20_000), random_state=SEED); xy = PCA(2, random_state=SEED).fit_transform(Pm.to_numpy())
dom = Pm.idxmax(axis=1).where(lambda s: s.isin(top12), "other")
sns.scatterplot(x=xy[:, 0], y=xy[:, 1], hue=dom.values, s=8, alpha=.6, ax=ax[1], palette="tab20", linewidth=0); ax[1].set(title="PCA of user interest vectors, coloured by dominant category", xlabel="PC1", ylabel="PC2")
ax[1].legend(fontsize=7, ncol=2, markerscale=2)
savefig("interest_representation")
pca = PCA(min(10, Pm.shape[1]), random_state=SEED).fit(Pm.to_numpy()); print("explained variance (first components):", pca.explained_variance_ratio_.round(3))

## 16 · Findings (fill in as a team)
Use these prompts for the report; every claim should point at a figure in `eda/figures/`.

1. **Behaviour mix:** which actions are rare, and what does P(action | click) say about depth of engagement? → `03_action_rates`, `03_action_cooccurrence`
2. **Inequality:** Gini of users/items, popularity bias, and what that means for a popularity baseline → `03_user_activity`, `03_item_popularity`
3. **Interest concentration:** how concentrated is a typical user on 1–3 categories? → `03_interest_concentration`
4. **Time:** peak hours, weekday/weekend differences, and whether engagement quality changes across the day → `03_temporal_patterns`
5. **Recency & repeats:** is the decay in `03_repeat_novelty_recency` strong enough to keep `HALF_LIFE_DAYS`?
6. **Search:** does the lift table in section 11 support `SEARCH_BONUS`?
7. **Viability:** best single-feature AUC and which feature group is strongest → `03_feature_viability`
8. **Risks for the model:** cold-start share, drift between splits, 19-day horizon limits → `03_coldstart_drift`

**Decisions to record:** intent weights, lookback/half-life, split fractions, k-core thresholds, any categories dropped, and the baselines to beat (popularity, equal-weight interactions).